# Case Machine Learning Engineer Sênior: Parte 1
## Visão Geral do Projeto

Este projeto implementa um pipeline de extração, processamento e ingestão de dados utilizando uma abordagem assíncrona e orientada a streaming. Os dados são extraídos da PokeAPI por meio de requisições paralelas controladas, processados em batches e persistidos em tabelas Delta Lake no Databricks.

A arquitetura adotada prioriza eficiência, escalabilidade e governança, incorporando metadados de extração para rastreabilidade e permitindo rollback lógico em caso de falhas. O pipeline integra processamento assíncrono em Python com o modelo Lakehouse, garantindo ingestão incremental, controle de recursos e consistência dos dados.

## Instalando e importando as bibliotecas necessárias para a extração

In [0]:
%pip install httpx aiostream pydantic
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from typing import AsyncGenerator
from httpx import AsyncClient
from pydantic import BaseModel, Field, field_validator
from aiostream import stream
from itertools import count
import pyspark.sql.functions as F
from pyspark.sql import Window
from datetime import datetime, timezone

## Definindo os modelos de dados e validação de input

In [0]:
class Stats(BaseModel):
    stat_name: str
    base_stat: int

    @field_validator("stat_name", mode="before")
    @classmethod
    def extract_stat_name(cls, v):
        return v["name"] if isinstance(v, dict) else v


class Types(BaseModel):
    type_name: str

    @field_validator("type_name", mode="before")
    @classmethod
    def extract_type_name(cls, v):
        return v["name"] if isinstance(v, dict) else v


class Abilities(BaseModel):
    ability_name: str
    is_hidden: bool

    @field_validator("ability_name", mode="before")
    @classmethod
    def extract_ability_name(cls, v):
        return v["name"] if isinstance(v, dict) else v


class Pokemon(BaseModel):
    id: int
    name: str
    types: list[Types]
    abilities: list[Abilities]
    stats: list[Stats]
    height: float
    weight: float
    base_experience: int | None

    @field_validator("types", mode="before")
    @classmethod
    def parse_types(cls, v):
        return [{"type_name": item["type"]} for item in v]

    @field_validator("abilities", mode="before")
    @classmethod
    def parse_abilities(cls, v):
        return [
            {"ability_name": item["ability"], "is_hidden": item["is_hidden"]}
            for item in v
        ]

    @field_validator("stats", mode="before")
    @classmethod
    def parse_stats(cls, v):
        return [
            {"stat_name": item["stat"], "base_stat": item["base_stat"]} for item in v
        ]

    def dump_pokemon(self) -> dict:
        return [
            {
                "pokemon_id": self.id,
                "name": self.name,
                "height": self.height,
                "weight": self.weight,
                "base_experience": self.base_experience,
            }
        ]

    def dump_types(self) -> dict:
        return [
            {
                "pokemon_id": self.id,
                "type_name": type.type_name,
            }
            for type in self.types
        ]

    def dump_stats(self) -> dict:
        return [
            {
                "pokemon_id": self.id,
                "stat_name": stat.stat_name,
                "base_stat": stat.base_stat,
            }
            for stat in self.stats
        ]

    def dump_abilities(self) -> dict:
        return [
            {
                "pokemon_id": self.id,
                "ability_name": ability.ability_name,
                "is_hidden": ability.is_hidden,
            }
            for ability in self.abilities
        ]

## Definição do Esquema Lógico e Pipeline de Extração no Databricks

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pokemon (
    pokemon_id              BIGINT,
    name            STRING,
    height          DOUBLE,
    weight          DOUBLE,
    base_experience BIGINT,

    extracted_at    TIMESTAMP
)
USING DELTA
PARTITIONED BY (extracted_at);

CREATE TABLE IF NOT EXISTS pokemon_stats (
    pokemon_id   BIGINT,
    stat_name    STRING,
    base_stat    BIGINT,

    extracted_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (extracted_at);

CREATE TABLE IF NOT EXISTS pokemon_type (
    pokemon_id   BIGINT,
    type_name    STRING,

    extracted_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (extracted_at);

CREATE TABLE IF NOT EXISTS pokemon_ability (
    pokemon_id   BIGINT,
    ability_name STRING,
    is_hidden    BOOLEAN,

    extracted_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (extracted_at);

## Criação do Client de Extração

Para a implementação do client de extração, foi utilizada a biblioteca **httpx**, por meio do `AsyncClient`, permitindo a execução de requisições assíncronas e a introdução de paralelismo controlado, reduzindo significativamente o tempo total de extração.

Além disso, foi empregada a biblioteca **aiostream**, que possibilita o gerenciamento do fluxo de requisições em um modelo orientado a *streaming*. Essa abordagem facilita a aplicação de mecanismos de controle, como a limitação da quantidade de requisições simultâneas, garantindo maior previsibilidade no consumo de recursos e evitando sobrecarga na API de origem.

Com essa arquitetura, torna-se simples definir limites para a extração paralela de dados — por exemplo, restringindo o número máximo de requisições concorrentes para a obtenção das informações dos Pokémon — mantendo eficiência, escalabilidade e controle sobre o processo de ingestão.

In [0]:
class PokeApiClient():
    def __init__(self):
        self.client = AsyncClient()
        self.list_limit = 20
        self.parallel_information_limit = 10
        self.batch_size = 500

    async def list_pokemons(self) -> AsyncGenerator[None, str]:
        for offset in count(0, self.list_limit):
            print(f"Valor do offset de listagem de nomes: {offset}")
            res = await self.client.get(f"https://pokeapi.co/api/v2/pokemon?offset={offset}&limit={self.list_limit}")
            data = res.json()
            if not data["results"]: break
            for item in data["results"]:
                yield item["name"]

    async def get_pokemon_info(self, name: int) -> Pokemon:
        response = await self.client.get(f"https://pokeapi.co/api/v2/pokemon/{name}")
        return Pokemon.model_validate(response.json())
    
    async def get_all_pokemons(self) -> AsyncGenerator[list[Pokemon], None]:
        names = self.list_pokemons()
        # Me permite limitar a quantidade de requisições simultâneas a rota de informações
        info_stream = stream.map(names, self.get_pokemon_info, task_limit=self.parallel_information_limit)

        batched_stream = stream.chunks(info_stream, self.batch_size)
        
        async with batched_stream.stream() as streamer:
            async for batch in streamer:
                yield batch

    async def close(self):
        await self.client.aclose()

In [0]:
poke_client = PokeApiClient()
extracted_at = datetime.now(timezone.utc)

try:
    async for batch in poke_client.get_all_pokemons():
        print(f"Salvando batch com: {len(batch)} pokemons.")
        partial_info = sum([pokemon.dump_pokemon() for pokemon in batch], [])
        spark.createDataFrame(partial_info).withColumn("extracted_at", F.lit(extracted_at)).write.mode("append").saveAsTable("pokemon")

        partial_stats = sum([pokemon.dump_stats() for pokemon in batch], [])
        spark.createDataFrame(partial_stats).withColumn("extracted_at", F.lit(extracted_at)).write.mode("append").saveAsTable("pokemon_stats")
        
        partial_types = sum([pokemon.dump_types() for pokemon in batch], [])
        spark.createDataFrame(partial_types).withColumn("extracted_at", F.lit(extracted_at)).write.mode("append").saveAsTable("pokemon_type")

        partial_abilities = sum([pokemon.dump_abilities() for pokemon in batch], [])
        
        spark.createDataFrame(partial_abilities).withColumn("extracted_at", F.lit(extracted_at)).write.mode("append").saveAsTable("pokemon_ability")
except Exception as e:
    print("Erro na extração, realizando rollback lógico...")

    spark.sql(f"DELETE FROM pokemon WHERE extracted_at = TIMESTAMP '{extracted_at}'")
    spark.sql(f"DELETE FROM pokemon_stats WHERE extracted_at = TIMESTAMP '{extracted_at}'")
    spark.sql(f"DELETE FROM pokemon_type WHERE extracted_at = TIMESTAMP '{extracted_at}'")
    spark.sql(f"DELETE FROM pokemon_ability WHERE extracted_at = TIMESTAMP '{extracted_at}'")

    raise e
finally:
    await poke_client.close()

Valor do offset de listagem de nomes: 0
Valor do offset de listagem de nomes: 20
Valor do offset de listagem de nomes: 40
Valor do offset de listagem de nomes: 60
Valor do offset de listagem de nomes: 80
Valor do offset de listagem de nomes: 100
Valor do offset de listagem de nomes: 120
Valor do offset de listagem de nomes: 140
Valor do offset de listagem de nomes: 160
Valor do offset de listagem de nomes: 180
Valor do offset de listagem de nomes: 200
Valor do offset de listagem de nomes: 220
Valor do offset de listagem de nomes: 240
Valor do offset de listagem de nomes: 260
Valor do offset de listagem de nomes: 280
Valor do offset de listagem de nomes: 300
Valor do offset de listagem de nomes: 320
Valor do offset de listagem de nomes: 340
Valor do offset de listagem de nomes: 360
Valor do offset de listagem de nomes: 380
Valor do offset de listagem de nomes: 400
Valor do offset de listagem de nomes: 420
Valor do offset de listagem de nomes: 440
Valor do offset de listagem de nomes: 46

## Diagrama do Banco de Dados

<div style="display: flex; justify-content: center;">
  <img src="./Untitled.png" alt="Diagrama do Banco de Dados" width="700"/>
</div>

<p style="text-align: center;"><em>Figura – Modelo lógico das tabelas utilizadas no pipeline de ingestão.</em></p>


## Pergunta 1: Quantos pokémons possuem mais de um type_name e têm a força maior que a média geral de todos os pokémons?
Definição de força: soma de todos os base_stat de cada pokémon.

In [0]:
pokemon_strength = (
    spark.table("pokemon_stats")
    .groupBy("pokemon_id")
    .agg(F.sum("base_stat").alias("strength"))
)
mean_strength = (
    pokemon_strength
    .agg(F.avg("strength").alias("mean_strength"))
    .collect()[0]["mean_strength"]
)
pokemon_multiple_types = (
    spark.table("pokemon_type")
    .groupBy("pokemon_id")
    .count()
    .filter(F.col("count") > 1)
    .select("pokemon_id")
)
strong_multi_type_pokemon = (
    pokemon_strength
    .join(pokemon_multiple_types, "pokemon_id")
    .filter(F.col("strength") > mean_strength)
)
result = strong_multi_type_pokemon.count()
display(result)

510

## Pergunta 2: Quais habilidades não aparecem em nenhum pokémon de tipo único?

In [0]:
pokemon_single_type = (
    spark.table("pokemon_type")
    .groupBy("pokemon_id")
    .count()
    .filter(F.col("count") == 1)
    .select("pokemon_id")
)
abilities_from_single_type = (
    spark.table("pokemon_ability")
    .join(pokemon_single_type, "pokemon_id")
    .select("ability_name")
    .distinct()
)
all_abilities = (
    spark.table("pokemon_ability")
    .select("ability_name")
    .distinct()
)
result = (
    all_abilities
    .join(
        abilities_from_single_type,
        on="ability_name",
        how="left_anti"
    )
)
display(result)

ability_name
dancer
merciless
corrosion
punk-rock
protosynthesis
aura-break
misty-surge
defeatist
water-bubble
shields-down


## Pergunta 3: Quais são os 5 pokémons que apresentam maior versatilidade de acordo com este critério?
A métrica de cálculo estabelecida para o índice de versatilidade é a seguinte:
versatility_score = (número de tipos * 2) + (número de abilities) + (somados stats / 100)

In [0]:
pokemon_total_stats = (
    spark.table("pokemon_stats")
    .groupBy("pokemon_id")
    .agg(F.sum("base_stat").alias("total_stats"))
)
pokemon_type_count = (
    spark.table("pokemon_type")
    .groupBy("pokemon_id")
    .count()
    .withColumnRenamed("count", "amount_types")
)
pokemon_ability_count = (
    spark.table("pokemon_ability")
    .groupBy("pokemon_id")
    .count()
    .withColumnRenamed("count", "amount_abilities")
)
pokemon_merged = (
    spark.table("pokemon")
    .select("pokemon_id", "name")
    .join(pokemon_total_stats, "pokemon_id")
    .join(pokemon_type_count, "pokemon_id")
    .join(pokemon_ability_count, "pokemon_id")
)
pokemon_versatility = (
    pokemon_merged
    .withColumn(
        "versatility_score",
        F.col("total_stats") / 100
        + F.col("amount_types") * 2
        + F.col("amount_abilities")
    )
)
result_1 = (
    pokemon_versatility
    .select("name", "versatility_score")
    .orderBy(F.col("versatility_score").desc())
    .limit(5)
)
display(result_1)

name,versatility_score
eternatus-eternamax,16.25
dragapult,13.0
kommo-o,13.0
goodra-hisui,13.0
archaludon,13.0


#### Poderia ter parado por aqui, mas isso é um resultado ingenuo, já que mais de 4 pokemons possuem 13 de versartily_score, ou seja um top 5 de fato é "injusto". Uma forma de tentar mitigar esse problema é trazer o top 5 maiores versatility_score.

In [0]:
window = Window.orderBy(F.col("versatility_score").desc())

ranked = (
    pokemon_versatility
    .select("name", "versatility_score")
    .withColumn("rank", F.dense_rank().over(window))
)

result = (
    ranked
    .filter(F.col("rank") <= 5)
    .orderBy(F.col("versatility_score").desc())
)
display(result)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


name,versatility_score,rank
eternatus-eternamax,16.25,1
kommo-o,13.0,2
dragapult,13.0,2
archaludon,13.0,2
kommo-o-totem,13.0,2
goodra-hisui,13.0,2
lugia,12.8,3
ho-oh,12.8,3
dialga,12.8,3
palkia,12.8,3
